In [ ]:
!pip install torch-optimizer --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 6.1 MB/s eta 0:00:00


In [ ]:
import os
import zipfile
import shutil
import numpy as np
import cv2
from glob import glob
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import KFold
import math

# =========================
# 1. Extract Dataset
# =========================
zip_path = "/content/stage1_train.zip"
extract_path = "/content/stage1_train"

if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ stage1_train.zip extracted.")

# =========================
# 2. Dataset Class
# =========================
class CellNucleiDataset(Dataset):
    def __init__(self, image_ids, image_size=(256, 256), augment=False):
        self.image_ids = image_ids
        self.image_size = image_size
        self.augment = augment
        self.transform = A.Compose([
            A.HorizontalFlip(),
            A.VerticalFlip(),
            A.RandomBrightnessContrast(),
            A.GaussianBlur(blur_limit=3),
            A.ShiftScaleRotate(0.1, 0.1, 15, p=0.5),
            A.GridDistortion(p=0.5),
            A.CoarseDropout(holes=8, max_h_size=16, max_w_size=16, p=0.5),
            A.Resize(*image_size),
            ToTensorV2()
        ]) if augment else A.Compose([
            A.Resize(*image_size),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = os.path.join(img_id, "images", os.path.basename(img_id) + ".png")
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = np.zeros(img.shape[:2], dtype=np.uint8)
        for m_path in glob(os.path.join(img_id, "masks", "*.png")):
            m = cv2.imread(m_path, 0)
            mask = np.maximum(mask, m)

        transformed = self.transform(image=img, mask=mask)
        image = transformed["image"].float() / 255.0
        mask = transformed["mask"].unsqueeze(0).float() / 255.0
        return image, mask

# =========================
# 3. Model with SELU Activation
# =========================
class SELU(nn.Module):
    def forward(self, x):
        return nn.functional.selu(x)

class ResidualConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.same_channels = in_ch == out_ch
        self.residual_conv = nn.Identity() if self.same_channels else nn.Conv2d(in_ch, out_ch, 1)
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            SELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch)
        )
        self.act = SELU()

    def forward(self, x):
        residual = self.residual_conv(x)
        x = self.conv(x)
        return self.act(x + residual)

class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        psi = self.relu(self.W_g(g) + self.W_x(x))
        psi = self.psi(psi)
        return x * psi

class AttentionResUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        filters = [64, 128, 256, 512, 1024]

        self.pool = nn.MaxPool2d(2)
        self.conv1 = ResidualConvBlock(in_ch, filters[0])
        self.conv2 = ResidualConvBlock(filters[0], filters[1])
        self.conv3 = ResidualConvBlock(filters[1], filters[2])
        self.conv4 = ResidualConvBlock(filters[2], filters[3])
        self.conv5 = ResidualConvBlock(filters[3], filters[4])

        self.up4 = nn.ConvTranspose2d(filters[4], filters[3], 2, 2)
        self.att4 = AttentionBlock(filters[3], filters[3], filters[2])
        self.up_conv4 = ResidualConvBlock(filters[4], filters[3])

        self.up3 = nn.ConvTranspose2d(filters[3], filters[2], 2, 2)
        self.att3 = AttentionBlock(filters[2], filters[2], filters[1])
        self.up_conv3 = ResidualConvBlock(filters[3], filters[2])

        self.up2 = nn.ConvTranspose2d(filters[2], filters[1], 2, 2)
        self.att2 = AttentionBlock(filters[1], filters[1], filters[0])
        self.up_conv2 = ResidualConvBlock(filters[2], filters[1])

        self.up1 = nn.ConvTranspose2d(filters[1], filters[0], 2, 2)
        self.up_conv1 = ResidualConvBlock(filters[1], filters[0])

        self.final = nn.Conv2d(filters[0], out_ch, 1)

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv2(self.pool(x1))
        x3 = self.conv3(self.pool(x2))
        x4 = self.conv4(self.pool(x3))
        x5 = self.conv5(self.pool(x4))

        d4 = self.up4(x5)
        d4 = self.up_conv4(torch.cat([self.att4(d4, x4), d4], dim=1))

        d3 = self.up3(d4)
        d3 = self.up_conv3(torch.cat([self.att3(d3, x3), d3], dim=1))

        d2 = self.up2(d3)
        d2 = self.up_conv2(torch.cat([self.att2(d2, x2), d2], dim=1))

        d1 = self.up1(d2)
        d1 = self.up_conv1(torch.cat([x1, d1], dim=1))

        return self.final(d1)

# =========================
# 4. Loss Functions
# =========================
class DiceLoss(nn.Module):
    def forward(self, inputs, targets, eps=1e-7):
        inputs = torch.sigmoid(inputs).view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        return 1 - (2 * intersection + eps) / (inputs.sum() + targets.sum() + eps)

class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.5, beta=0.5, gamma=1.33, eps=1e-7):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.eps = eps

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs).view(-1)
        targets = targets.view(-1)
        TP = (inputs * targets).sum()
        FP = ((1 - targets) * inputs).sum()
        FN = (targets * (1 - inputs)).sum()
        tversky = (TP + self.eps) / (TP + self.alpha * FP + self.beta * FN + self.eps)
        return (1 - tversky) ** self.gamma

class HybridLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice = DiceLoss()
        self.ft = FocalTverskyLoss()

    def forward(self, inputs, targets):
        return 0.7 * self.ft(inputs, targets) + 0.3 * self.dice(inputs, targets)

# =========================
# 5. Training Utilities
# =========================
class EarlyStopping:
    def __init__(self, patience):
        self.patience = patience
        self.counter = 0
        self.best_loss = float("inf")
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
        if self.counter >= self.patience:
            self.early_stop = True

def cosine_annealing(epoch, total_epochs=100, initial_lr=1e-2):
    return initial_lr * (1 + math.cos(math.pi * epoch / total_epochs)) / 2

In [ ]:
import torch
import math
import os
import torch_optimizer as optim  # pip install torch-optimizer

# -------------------------
# Dice Metric
# -------------------------
def dice_score(preds, targets, threshold=0.5, smooth=1e-6):
    preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()
    intersection = (preds * targets).sum()
    return (2. * intersection + smooth) / (preds.sum() + targets.sum() + smooth)

# -------------------------
# Training Function
# -------------------------
def train_one_fold(model, train_loader, val_loader, fold, device, epochs=50, patience=10, initial_lr=1e-4, save_dir="/content/saved_models"):
    os.makedirs(save_dir, exist_ok=True)

    model = model.to(device)
    optimizer = optim.RAdam(model.parameters(), lr=initial_lr)
    loss_fn = HybridLoss()   # <-- already defined in your code
    early_stopper = EarlyStopping(patience=patience)

    best_val_loss = float("inf")
    best_model_path = os.path.join(save_dir, f"attention_resunet_fold{fold}.pth")

    for epoch in range(epochs):
        # Cosine annealing LR
        lr = initial_lr * (1 + math.cos(math.pi * epoch / epochs)) / 2
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # ---- Training ----
        model.train()
        train_loss = 0.0
        for img, mask in train_loader:
            img, mask = img.to(device), mask.to(device)
            optimizer.zero_grad()
            pred = model(img)
            loss = loss_fn(pred, mask)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # ---- Validation ----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for img, mask in val_loader:
                img, mask = img.to(device), mask.to(device)
                pred = model(img)
                val_loss += loss_fn(pred, mask).item()

        print(f"📘 Fold {fold} | Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_model_path)

        # Early stopping
        early_stopper(val_loss)
        if early_stopper.early_stop:
            print(f"⛔ Early Stopping triggered at Epoch {epoch+1}")
            break

    # ---- Final Dice Evaluation on Validation Set ----
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    model.eval()
    dice_total = 0.0
    n_batches = 0
    with torch.no_grad():
        for img, mask in val_loader:
            img, mask = img.to(device), mask.to(device)
            pred = model(img)
            dice_total += dice_score(pred, mask).item()
            n_batches += 1

    fold_dice = dice_total / n_batches
    print(f"🎯 Fold {fold} Dice: {fold_dice:.4f}")
    return fold_dice


In [ ]:
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader
import numpy as np

all_ids = sorted(glob("/content/stage1_train/*"))  # use the entire dataset
kf = KFold(n_splits=5, shuffle=True, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fold_dice_scores = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(all_ids), 1):
    print(f"\n--- Fold {fold} ---")
    tr_ids = [all_ids[i] for i in tr_idx]
    val_ids = [all_ids[i] for i in val_idx]

    train_ds = CellNucleiDataset(tr_ids, augment=True)
    val_ds   = CellNucleiDataset(val_ids, augment=False)

    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False)

    model = AttentionResUNet()
    fold_dice = train_one_fold(model, train_loader, val_loader, fold, device)
    fold_dice_scores.append(fold_dice)

# ---- Final Results ----
mean_dice = np.mean(fold_dice_scores)
std_dice = np.std(fold_dice_scores)

print("\n============================")
print(f"📊 Dice per Fold: {fold_dice_scores}")
print(f"✅ Mean Dice: {mean_dice:.4f} ± {std_dice:.4f}")
print("============================")


--- Fold 1 ---


/tmp/ipython-input-2787715530.py:45: UserWarning: Argument(s) 'holes, max_h_size, max_w_size' are not valid for transform CoarseDropout
  A.CoarseDropout(holes=8, max_h_size=16, max_w_size=16, p=0.5),


📘 Fold 1 | Epoch 1/50 | Train Loss: 42.7900 | Val Loss: 10.5679
📘 Fold 1 | Epoch 2/50 | Train Loss: 28.5598 | Val Loss: 6.0571
📘 Fold 1 | Epoch 3/50 | Train Loss: 23.7682 | Val Loss: 3.7993
📘 Fold 1 | Epoch 4/50 | Train Loss: 20.0069 | Val Loss: 10.2077
📘 Fold 1 | Epoch 5/50 | Train Loss: 15.8684 | Val Loss: 2.6587
📘 Fold 1 | Epoch 6/50 | Train Loss: 12.3226 | Val Loss: 1.7005
📘 Fold 1 | Epoch 7/50 | Train Loss: 11.3258 | Val Loss: 2.1955
📘 Fold 1 | Epoch 8/50 | Train Loss: 8.6339 | Val Loss: 1.3308
📘 Fold 1 | Epoch 9/50 | Train Loss: 7.2459 | Val Loss: 1.2807
📘 Fold 1 | Epoch 10/50 | Train Loss: 7.7712 | Val Loss: 1.2622
📘 Fold 1 | Epoch 11/50 | Train Loss: 8.4982 | Val Loss: 1.2977
📘 Fold 1 | Epoch 12/50 | Train Loss: 8.2316 | Val Loss: 1.2386
📘 Fold 1 | Epoch 13/50 | Train Loss: 8.6956 | Val Loss: 1.2258
📘 Fold 1 | Epoch 14/50 | Train Loss: 8.0532 | Val Loss: 1.1557
📘 Fold 1 | Epoch 15/50 | Train Loss: 7.3884 | Val Loss: 1.8716
📘 Fold 1 | Epoch 16/50 | Train Loss: 8.3058 | Val Loss: